# Optimization for Deep Learning

This notebook makes the units of a practical training run explicit: dataset
indices, shuffled minibatches, optimizer steps, and epochs. It then trains the
same small network with several PyTorch optimizers and inspects their state.

The data are the local scikit-learn handwritten digits used in Lecture 12.

In [1]:
from pathlib import Path
import math
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 26513
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)

digits = pd.read_csv('SklearnDigits.csv')
X = torch.tensor(digits.drop(columns='target').to_numpy(np.float32) / 16)
y = torch.tensor(digits.target.to_numpy(np.int64))
train_X, valid_X = X[:1400], X[1400:]
train_y, valid_y = y[:1400], y[1400:]
print('training:', train_X.shape, train_y.shape)
print('validation:', valid_X.shape, valid_y.shape)

training: torch.Size([1400, 64]) torch.Size([1400])
validation: torch.Size([397, 64]) torch.Size([397])


## Dataset, shuffled batches, and epochs

`Dataset` defines an indexed collection. `DataLoader` decides the order and
groups indices into batches. Iterating through the loader once is one epoch.

In [2]:
train_dataset = TensorDataset(train_X, train_y)
BATCH_SIZE = 128
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    generator=generator,
)

print('steps per epoch:', len(train_loader))
print('formula:', math.ceil(len(train_dataset) / BATCH_SIZE))
for step, (xb, yb) in enumerate(train_loader):
    if step in [0, len(train_loader) - 1]:
        print(f'step {step + 1:2d}: X {tuple(xb.shape)}, y {tuple(yb.shape)}')
assert len(train_loader) == math.ceil(len(train_dataset) / BATCH_SIZE)

steps per epoch: 11
formula: 11
step  1: X (128, 64), y (128,)
step 11: X (120, 64), y (120,)


In [3]:
# A sampler view: attach each observation's index as an extra tensor.
index_dataset = TensorDataset(torch.arange(20), torch.arange(20))
index_loader = DataLoader(
    index_dataset,
    batch_size=6,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
for batch_number, (indices, _) in enumerate(index_loader, start=1):
    print(f'batch {batch_number}:', indices.tolist())

batch 1: [3, 10, 9, 1, 13, 12]
batch 2: [5, 2, 18, 8, 7, 4]
batch 3: [14, 6, 16, 17, 11, 0]
batch 4: [15, 19]


## One complete PyTorch training procedure

The function below reports both epoch and optimizer-step counts. The scheduler
is stepped after each optimizer update, so its unit is optimizer steps.

In [4]:
def make_model():
    torch.manual_seed(SEED)
    return nn.Sequential(
        nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)
    )

def accuracy(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        return float((model(inputs).argmax(1) == targets).float().mean())

def train(optimizer_name, epochs=12):
    model = make_model()
    if optimizer_name == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=0.08, momentum=0.9)
    elif optimizer_name == 'AdaGrad':
        optimizer = torch.optim.Adagrad(model.parameters(), lr=0.04)
    elif optimizer_name == 'RMSProp':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=0.002, alpha=0.99)
    elif optimizer_name == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-3)
    else:
        raise ValueError(optimizer_name)

    # Recreate the loader so all methods receive the same seeded shuffles.
    batches = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    total_steps = epochs * len(batches)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()
    rows, optimizer_step = [], 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in batches:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer_step += 1
            total_loss += float(loss) * len(yb)
        rows.append({
            'optimizer': optimizer_name,
            'epoch': epoch,
            'optimizer_step': optimizer_step,
            'training_loss': total_loss / len(train_dataset),
            'validation_accuracy': accuracy(model, valid_X, valid_y),
            'learning_rate': scheduler.get_last_lr()[0],
        })
    return model, optimizer, pd.DataFrame(rows)

model, optimizer, adamw_history = train('AdamW')
adamw_history.tail()

/var/folders/dw/twkl7xz54f59zmk2mt5tr7t40000gn/T/ipykernel_18746/2087274754.py:46: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  total_loss += float(loss) * len(yb)


,optimizer,epoch,optimizer_step,training_loss,validation_accuracy,learning_rate
7,AdamW,8,88,0.856926,0.836272,0.000500
8,AdamW,9,99,0.803347,0.836272,0.000293
9,AdamW,10,110,0.773539,0.836272,0.000134
10,AdamW,11,121,0.760008,0.841310,0.000034
11,AdamW,12,132,0.755804,0.841310,0.000000


## Optimizer state is persistent memory

AdamW stores first- and second-moment tensors for each trained parameter.
Momentum SGD stores one momentum buffer. Plain SGD would store none.

In [5]:
parameter_count = sum(p.numel() for p in model.parameters())
state_tensors = [
    value for state in optimizer.state.values() for value in state.values()
    if torch.is_tensor(value) and value.ndim > 0
]
state_count = sum(value.numel() for value in state_tensors)
print('parameters:', parameter_count)
print('AdamW moment entries:', state_count)
print('state / parameters:', state_count / parameter_count)
assert state_count == 2 * parameter_count

parameters: 4810
AdamW moment entries: 9620
state / parameters: 2.0


## Compare optimizers under one declared protocol

This is a teaching comparison, not a leaderboard. The initialization, batch
orders, batch size, epoch budget, and cosine schedule are held fixed, while the
base learning rate is specified in the training function for each optimizer.

In [6]:
histories = []
for name in ['SGD', 'AdaGrad', 'RMSProp', 'AdamW']:
    _, _, history = train(name)
    histories.append(history)
comparison = pd.concat(histories, ignore_index=True)
comparison.groupby('optimizer').tail(1)[
    ['optimizer', 'training_loss', 'validation_accuracy', 'optimizer_step']
].sort_values('validation_accuracy', ascending=False)

,optimizer,training_loss,validation_accuracy,optimizer_step
11,SGD,0.150991,0.889169,132
23,AdaGrad,0.175674,0.874055,132
35,RMSProp,0.261591,0.874055,132
47,AdamW,0.755804,0.841310,132
